# 01 · Ingestão — Camada Bronze

Neste notebook são ingeridos os dados brutos na camada Bronze. Os arquivos CSV 
são lidos sem nenhuma alteração de conteúdo ou estrutura e gravados em formato 
Delta Lake no modo Append. É adicionada a coluna `ingestion_datetime` como 
metadado de auditoria, registrando o momento exato da gravação.

Além dos arquivos CSV, é realizada a ingestão da cotação do dólar (PTAX) 
diretamente da API do Banco Central do Brasil, com datas parametrizadas via widgets.


In [0]:
catalog = "cinedata_analytics"
bronze_schema = f"{catalog}.bronze"
volume_name = "inputs"
landing_path = f"/Volumes/{catalog}/bronze/{volume_name}"

print("Conectado ao schema Bronze:", bronze_schema)
print("Lendo arquivos de:", landing_path)

Conectado ao schema Bronze: cinedata_analytics.bronze
Lendo arquivos de: /Volumes/cinedata_analytics/bronze/inputs


## Leitura dos Arquivos CSV

Os 5 arquivos são lidos a partir do Volume `inputs` com `inferSchema=False`, 
ou seja, todos os campos são mantidos como texto puro. Essa escolha evita que 
valores sujos ou inesperados sejam interpretados de forma incorreta pelo Spark 
antes do tratamento na camada Silver.

In [0]:
from pyspark.sql.functions import current_timestamp

path_info = f"{landing_path}/movies_info_TMDB_IMDB.csv"
path_financials = f"{landing_path}/movies_financials_IMDB_TMDB.csv"
path_metrics = f"{landing_path}/movies_metrics_IMDB_TMDB.csv"
path_credits = f"{landing_path}/credits_and_tags_IMDB_TMDB.csv"
path_reviews = f"{landing_path}/movies_reviews.csv"

df_info_raw = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "false") \
    .option("quote", "\"") \
    .option("escape", "\"") \
    .load(path_info)

df_financials_raw = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "false") \
    .load(path_financials)

df_metrics_raw = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "false") \
    .option("quote", "\"") \
    .option("escape", "\\") \
    .load(path_metrics)

df_credits_raw = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "false") \
    .option("quote", "\"") \
    .option("escape", "\"") \
    .load(path_credits)

df_reviews_raw = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "false") \
    .load(path_reviews)

## Gravação das Tabelas Bronze em Delta (Modo Append)

Cada DataFrame é gravado como tabela Delta com o modo `Append`, garantindo que 
execuções futuras acumulem o histórico em vez de sobrescrever os dados anteriores.

In [0]:
df_info_raw \
    .withColumn("ingestion_datetime", current_timestamp()) \
    .write.format("delta").mode("append") \
    .saveAsTable(f"{bronze_schema}.tb_movies_info")

df_financials_raw \
    .withColumn("ingestion_datetime", current_timestamp()) \
    .write.format("delta").mode("append") \
    .saveAsTable(f"{bronze_schema}.tb_movies_financials")

df_metrics_raw \
    .withColumn("ingestion_datetime", current_timestamp()) \
    .write.format("delta").mode("append") \
    .saveAsTable(f"{bronze_schema}.tb_movies_metrics")

df_credits_raw \
    .withColumn("ingestion_datetime", current_timestamp()) \
    .write.format("delta").mode("append") \
    .saveAsTable(f"{bronze_schema}.tb_credits_and_tags")

df_reviews_raw \
    .withColumn("ingestion_datetime", current_timestamp()) \
    .write.format("delta").mode("append") \
    .saveAsTable(f"{bronze_schema}.tb_movies_reviews")

display(spark.table(f"{bronze_schema}.tb_movies_info").limit(5))

id,tconst,title,original_title,original_language,release_date,runtime,status,overview,tagline,ingestion_datetime
293660,tt1431045,Deadpool,Deadpool,en,2016-02-09,108,Released,"The origin story of former Special Forces operative turned mercenary Wade Wilson, who, after being subjected to a rogue experiment that leaves him with accelerated healing powers, adopts the alter ego Deadpool. Armed with his new abilities and a dark, twisted sense of humor, Deadpool hunts down the man who nearly destroyed his life.",Witness the beginning of a happy ending.,2026-10-06T18:29:22.974Z
299536,tt4154756,AVENGERS: INFINITY WAR,Avengers: Infinity War,en,04-25-2018,149,Released,"As the Avengers and their allies have continued to protect the world from threats too large for any one hero to handle, a new danger has emerged from the cosmic shadows: Thanos. A despot of intergalactic infamy, his goal is to collect all six Infinity Stones, artifacts of unimaginable power, and use them to inflict his twisted will on all of reality. Everything the Avengers have fought for has led up to this moment - the fate of Earth and existence itself has never been more uncertain.",An entire universe. Once and for all.,2026-10-06T18:29:22.974Z
299534,tt4154796,Avengers: Endgame,Avengers: Endgame,en,2019-04-24,181,released,"After the devastating events of Avengers: Infinity War, the universe is in ruins due to the efforts of the Mad Titan, Thanos. With the help of remaining allies, the Avengers must assemble once more in order to undo Thanos' actions and restore order to the universe once and for all, no matter what consequences may be in store.",Avenge the fallen.,2026-10-06T18:29:22.974Z
475557,tt7286456,Joker,Joker,en,2019-10-01,122,Released,"During the 1980s, a failed stand-up comedian is driven insane and turns to a life of crime and chaos in Gotham City while becoming an infamous psychopathic crime figure.",Put on a happy face.,2026-10-06T18:29:22.974Z
271110,tt3498820,Captain America: Civil War,Captain America: Civil War,en,2016-04-27,147,Released,"Following the events of Age of Ultron, the collective governments of the world pass an act designed to regulate all superhuman activity. This polarizes opinion amongst the Avengers, causing two factions to side with Iron Man or Captain America, which causes an epic battle between former allies.",United we stand. Divided we fall.,2026-10-06T18:29:22.974Z


## Ingestão da Cotação do Dólar via API do Banco Central

A cotação de compra do dólar (PTAX) é consultada diretamente da API pública do 
Banco Central para o período definido nos widgets. O resultado é transformado em 
um DataFrame Spark e gravado na tabela `bronze.tb_cotacao_dolar`.

A requisição foi feita utilizando o protocolo padrão do Python com cabeçalho explícito de User-Agent simulando navegação segura, apontando diretamente para o serviço OData do Banco Central (olinda.bcb.gov.br) com o retorno forçado em $format=json, evitando os bloqueios automáticos de firewall do portal do governo.


In [0]:
import urllib.request
import json
from datetime import datetime, timedelta
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

hoje = datetime.now()
data_fim_padrao = hoje.strftime("%m-%d-%Y")
data_inicio_padrao = (hoje - timedelta(days=7)).strftime("%m-%d-%Y")

dbutils.widgets.text("data_inicio", data_inicio_padrao, "Data Início (MM-DD-AAAA)")
dbutils.widgets.text("data_fim", data_fim_padrao, "Data Fim (MM-DD-AAAA)")

data_inicio = dbutils.widgets.get("data_inicio").strip()
data_fim = dbutils.widgets.get("data_fim").strip()

print(f"Buscando cotação do dólar entre {data_inicio} e {data_fim}...")

url_bcb = (
    f"https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/CotacaoDolarPeriodo("
    f"dataInicial=@dataInicial,dataFinalCotacao=@dataFinalCotacao)?"
    f"@dataInicial='{data_inicio}'&@dataFinalCotacao='{data_fim}'&"
    f"$select=dataHoraCotacao,cotacaoCompra&$format=json"
)

requisicao = urllib.request.Request(url_bcb, headers={"User-Agent": "Mozilla/5.0"})
with urllib.request.urlopen(requisicao) as response:
    dados_api = json.loads(response.read().decode("utf-8"))

cotacoes = dados_api.get("value", [])
print(f"Total de cotações encontradas: {len(cotacoes)}")

schema_cotacao = StructType([
    StructField("dataHoraCotacao", StringType(), True),
    StructField("cotacaoCompra", DoubleType(), True)
])

df_cotacao_raw = spark.createDataFrame(cotacoes, schema=schema_cotacao)

df_cotacao_raw \
    .withColumn("ingestion_datetime", current_timestamp()) \
    .write.format("delta").mode("append") \
    .saveAsTable(f"{bronze_schema}.tb_cotacao_dolar")

print(f"Cotações salvas em {bronze_schema}.tb_cotacao_dolar.")

display(spark.table(f"{bronze_schema}.tb_cotacao_dolar").orderBy(current_timestamp().desc()).limit(10))


Buscando cotação do dólar entre 09-29-2026 e 10-06-2026...
Total de cotações encontradas: 6
Cotações salvas em cinedata_analytics.bronze.tb_cotacao_dolar.


dataHoraCotacao,cotacaoCompra,ingestion_datetime
2026-09-29 13:06:06.446305,5.2198,2026-10-06T18:29:48.918Z
2026-09-30 13:11:44.783262,5.1803,2026-10-06T18:29:48.918Z
2026-10-01 13:10:35.4469,5.2073,2026-10-06T18:29:48.918Z
2026-10-02 13:03:16.256632,5.2232,2026-10-06T18:29:48.918Z
2026-10-05 13:07:36.558602,4.9853,2026-10-06T18:29:48.918Z
2026-10-06 13:03:21.267287,4.9692,2026-10-06T18:29:48.918Z
2026-09-29 13:06:06.446305,5.2198,2026-10-07T13:27:16.326Z
2026-09-30 13:11:44.783262,5.1803,2026-10-07T13:27:16.326Z
2026-10-01 13:10:35.4469,5.2073,2026-10-07T13:27:16.326Z
2026-10-02 13:03:16.256632,5.2232,2026-10-07T13:27:16.326Z
